# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfahadkhan-max/Week-01-ML-FlyRank-AI-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
%pip install -q duckdb

import duckdb, os
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT * FROM glob('{BASE}/**/*.parquet') LIMIT 30").show()

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [21]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [22]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_clients.parquet') LIMIT 1")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES     │ NULL    │ NULL  

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (client_hash_id, content_hash_id, report_date) combination in fact_content_daily_performance — one piece of content, for one client, on one calendar day. Dev window: month=2026-03. month=2026-06 (_sample) is the sealed test month only.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature fields:** gsc_impressions, gsc_clicks, gsc_sum_position, day-of-week derived from report_date, gsc_data_available.

**Label field:** something derived from future gsc_clicks (e.g. "did clicks grow next period") — a proxy, not Google's real ranking signal.

**Context fields:** client_hash_id, content_hash_id, report_date — used to key/join/group, never fed to a model directly.

**Excluded, with why**: sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, and all ga4_* fields — excluded because my lane is GSC-only; mixing AI-referral and GA4 signals this early would hide which system any given signal came from.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain Check:**

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   n   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



**Row count + date span:**

In [26]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

┌─────────┬────────────┬────────────┐
│ n_rows  │ first_day  │  last_day  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



**Availability, IS TRUE:**

In [27]:
con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM((gsc_data_available) IS TRUE) AS rows_gsc_available,
      SUM((gsc_data_available IS TRUE) AND (gsc_impressions > 0)) AS rows_with_impressions
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬───────────────────────┐
│ total_rows │ rows_gsc_available │ rows_with_impressions │
│   int64    │       int128       │        int128         │
├────────────┼────────────────────┼───────────────────────┤
│    9841378 │            3611061 │               3611061 │
└────────────┴────────────────────┴───────────────────────┘



**Five features + the trap**

In [28]:
df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_sum_position
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE (gsc_data_available IS TRUE) AND (gsc_impressions > 0)
""").df()

df['avg_position'] = df['gsc_sum_position'] / df['gsc_impressions']
df['day_of_week'] = df['report_date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five features, one line each:

1.gsc_impressions — knowable same-day, logged by Search Console as the day closes.

2.avg_position — same-day GSC metric.

3.day_of_week — derived from the date itself, always known in advance.

4.is_weekend — same, calendar-derived.

5.gsc_data_available flag — known at logging time, not derived from the outcome.

**The trap:**

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df['label'] = (df['gsc_clicks'] > df['gsc_clicks'].median()).astype(int)

X_honest = df[['gsc_impressions','avg_position','day_of_week','is_weekend']].fillna(0)
y = df['label']
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=0)
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("Honest AUC:", roc_auc_score(yte, m.predict_proba(Xte)[:,1]))

df['leaky_clicks_rank'] = df['gsc_clicks'].rank(pct=True)  # derived straight from the label
X_leaky = X_honest.copy()
X_leaky['leaky_clicks_rank'] = df['leaky_clicks_rank']
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.2, random_state=0)
m2 = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("Leaky AUC:", roc_auc_score(yte, m2.predict_proba(Xte)[:,1]))  # jumps toward ~1.0

# delete the leak, keep the honest number
X_leaky = X_leaky.drop(columns=['leaky_clicks_rank'])

Honest AUC: 0.8473130395953404
Leaky AUC: 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

gsc_data_start in dim_clients varies per client, so this panel has an unbalanced history — some clients simply have no GSC rows in early months. This is not a random sample across time.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.